In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
class BatsmanState(TypedDict):
    balls:int
    runs:int
    fours:int
    sixes:int
    sr:float
    bpb:float
    boundary_percent:float
    summary:str

In [19]:
def calculate_sr(state:BatsmanState)->BatsmanState:
    sr=(state['runs']/state['balls'])*100
    
    return {'sr':sr} #rather than sending the entire state here we are just sending that state in which we want to update otherwise in parallel we will have conflicts

In [20]:
def calculate_bpb(state:BatsmanState)->BatsmanState:
    bpb=state['balls']/(state['fours']+state['sixes'])
    return {'bpb':bpb}

In [21]:
def caluculate_boundary_percent(state:BatsmanState)->BatsmanState:
    boundary_percent=(((state['fours']*4)+(state['sixes']*6))/state['runs'])*100
    return {'boundary_percent':boundary_percent}

In [22]:
def summary(state:BatsmanState)->BatsmanState:
    summary=f"""strike rate {state['sr']} \n
                Balls per boundary-{state['bpb']} \n
                Boundary percent-{state['boundary_percent']}
    """
    return {'summary':summary}

In [23]:
graph = StateGraph(BatsmanState)

# Adding nodes
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', caluculate_boundary_percent)
graph.add_node('summary', summary)

# Adding edges
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [24]:
initial_state={'runs':100,
               'balls':50,
               'fours':6,
               'sixes':4}

workflow.invoke(initial_state)

{'balls': 50,
 'runs': 100,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': 'strike rate 200.0 \n\n                Balls per boundary-5.0 \n\n                Boundary percent-48.0\n    '}